# Cuiman job result resources

Use the local testing service to discover STAC Assets, inspect and select portable
resources, compose a resolver for known folder contents, and open CSV data through
the unified `open_job_result()` API.

Use `list_job_result_resources()` to discover selectable targets and their opener
availability, then pass a resource to `open_job_result()`. Kind and immediate-member
views are available. Explicit remote traversal and continuation remain pending.

From the repository root, run `pixi install`, then keep `pixi run serve` running in
a separate terminal. Start Jupyter with `pixi run jl` and select the Pixi Python
environment. Server and notebook must share the same filesystem. Run cells in order;
if a cell fails, run the cleanup cell before starting again. The test processes
write into fresh temporary directories, removed by cleanup after reading. The
client uses a fresh profile path there rather than loading a personal profile.

See the [opener guide](../docs/cuiman/guides/openers.md) and
[resolver guide](../docs/cuiman/guides/resolvers.md) for the contracts and extension points.

In [ ]:
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

# Jupyter may start in the repository root or in notebooks/.
repo_root = Path.cwd()
if not (repo_root / "examples/guides/cuiman").is_dir():
    repo_root = repo_root.parent
if not (repo_root / "examples/guides/cuiman").is_dir():
    raise RuntimeError("Open this notebook from the Eozilla checkout")
sys.path.insert(0, str(repo_root))

import pandas as pd

from cuiman.api.resolver import MetadataLoader
from cuiman.api.resources import AmbiguousResourceError, JobResultResource
from examples.guides.cuiman.resolvers import (
    ProductFolderResolver,
    create_client,
    discover_output,
    fetch_local_metadata,
    submit_stac,
)

In [ ]:
output = TemporaryDirectory(prefix="cuiman-notebook-")
root = Path(output.name)
client = create_client(root / "client-config.yaml")
print("Temporary server-side output directory:", root)

## Discover an inline Item

`simulate_stac_item` returns an inline STAC Item in `result` and an independent
text Link in `report`. Submission creates one job; the helper waits for successful
completion before retrieving its original results. Discovery retains the original
output mapping and produces a separate flat view.

In [ ]:
job_id, results, process = await submit_stac(
    client, "simulate_stac_item", root / "inline"
)
print("Job:", job_id)
print("Original outputs:", list(results.root or {}))
print(results.model_dump_json(indent=2))

In [ ]:
resources = await client.list_job_result_resources(
    job_id, output_name="result", data_type=pd.DataFrame
)
resources

## Inspect, select, and open one Asset

The default listing contains three Assets: `data`, `report`, and `products`.
Relative Asset URLs use the inline Item's self link as their base. Missing roles
and media types remain unspecified. The CSV has an available pandas reader; this
is a candidate assessment without payload access. Preview availability stays unknown.

Rendering and selection use loaded descriptions only. Select the CSV Asset and
open it through the receiving client's built-in pandas reader. The resource form
performs no job lookup, polling, or repeated discovery. `output_name`,
`poll_interval`, and `timeout` are job-only arguments and must be omitted here.

In [ ]:
data = resources.select(key="data")
print(data.model_dump_json(indent=2))
table = await client.open_job_result(data, data_type=pd.DataFrame)
table

## Inspect outputs, Items, and immediate members

Omitting `output_name` includes the independent report. Kind selection exposes
Item containers; a parent selector returns their immediate loaded Assets.
These views never authorize remote Catalog/Collection crawling.


In [ ]:
all_outputs = await client.list_job_result_resources(job_id)
print("Output states:", dict(all_outputs.output_states))
items = await client.list_job_result_resources(
    job_id, output_name="result", kind="stac-item"
)
members = await client.list_job_result_resources(job_id, parent_id=items[0].id)
assert members.select(key="data").id == data.id
members

## Round-trip a portable snapshot and override reader settings

Resources are frozen descriptions. Nested JSON objects are read-only mappings and
arrays are tuples in Python, but serialize as ordinary JSON. A snapshot contains
no live client, loader, credential provider, or resolved storage credentials.

Reader defaults and validated producer hints are applied before client settings
and explicit call options. Here `dtype` is a trusted caller option forwarded to
pandas. Passing the restored resource leaves its advertised metadata unchanged.

In [ ]:
snapshot = data.model_dump_json()
restored = JobResultResource.model_validate_json(snapshot)
assert restored == data
restored_table = await client.open_job_result(
    restored, data_type=pd.DataFrame, dtype={"date": str}
)
pd.testing.assert_frame_equal(restored_table, table)
restored_table

## Add declared folder descendants

The products folder has no trailing slash. `ProductFolderResolver` reuses
`StacResolver` and owns a `FolderResourceTransformer` with a declared
`tables/observations.csv` entry. It applies only to the testing processes.
See the maintained guide source for its no-argument constructor and `ResourceEntry`
declarations.

The transformer keeps ancestors, gives the CSV its own format and roles, and
records configuration provenance. It does not scan storage or verify access.
Completeness refers to the declared configuration, not the entire directory.
Register it temporarily on this guide's isolated configuration class.

In [ ]:
unregister = client.config.register_job_result_resolver(ProductFolderResolver)
try:
    expanded = await client.list_job_result_resources(
        job_id, output_name="result", data_type=pd.DataFrame
    )
finally:
    unregister()
expanded

In [ ]:
configured = expanded.select(key="observations")
print("Configured selector:", configured.id)
print("Original Asset selector:", data.id)
print("Parent:", configured.parent_id)
print("Configuration:", configured.provenance["configuration_id"])
configured_table = await client.open_job_result(
    configured, data_type=pd.DataFrame
)
pd.testing.assert_frame_equal(configured_table, table)
configured_table

## Fetch a linked ItemCollection within discovery limits

The client reads bounded JSON metadata using unauthenticated HTTP(S) or local
files. Configure `ClientConfig.job_result_metadata_fetcher` for another transport
or scoped authentication. Processing API credentials are not forwarded to metadata
or Asset locations. No CSV payload is fetched during discovery.

For comparison, the lower-level helper below supplies a loader explicitly to the
developer resolver contract. That flat view includes containers and has no client
capability assessment. Both views preserve the same Asset selectors.


In [ ]:
linked_job_id, linked_results, linked_process = await submit_stac(
    client, "simulate_stac_item_collection", root / "linked"
)
print(linked_results.model_dump_json(indent=2))
loader = MetadataLoader(fetch_local_metadata)
flat = await discover_output(
    client, linked_job_id, linked_results, linked_process, loader=loader
)
print("Metadata fetches:", loader.request_count)
assert loader.request_count == 1
linked = await client.list_job_result_resources(
    linked_job_id, output_name="result", data_type=pd.DataFrame
)
assert flat.select(item_id="ndvi-2026-09-02", key="data").id == (
    linked.select(item_id="ndvi-2026-09-02", key="data").id
)
linked

## Use ownership to disambiguate repeated keys

Both Items have a `data` Asset, so key-only selection is ambiguous. Select by
Item ID and key, or use the opaque resource ID. A match refers only to currently
loaded resources; selection never searches additional pages. Limits and malformed
entries remain visible as diagnostics, while Catalog/Collection navigation and
advertised next pages are deferred.

In [ ]:
try:
    linked.select(key="data")
except AmbiguousResourceError as error:
    print(error)

second = linked.select(item_id="ndvi-2026-09-02", key="data")
assert linked.select(id=second.id) == second
second_table = await client.open_job_result(second, data_type=pd.DataFrame)
second_table

In [ ]:
again = await client.list_job_result_resources(
    linked_job_id, output_name="result", data_type=pd.DataFrame
)
assert again == linked
refreshed = await client.list_job_result_resources(
    linked_job_id, output_name="result", data_type=pd.DataFrame, refresh=True
)
assert [r.id for r in refreshed] == [r.id for r in linked]
print("Listings reuse metadata; refresh re-reads it. Each call has a fresh budget.")

## Cleanup

Close the client and remove only this notebook's temporary output directory after
all reads. The saved resource snapshots still describe their targets, but those
temporary files will no longer exist. Re-running submission cells creates new jobs.

The [opener guide](../docs/cuiman/guides/openers.md) also demonstrates
`open_job_result(job_id, output_name=...)` with an ordinary Zarr output and a custom
reader. That job-ID form retains polling and opens the original output; it does
not automatically select an Asset inside a STAC result.

In [ ]:
try:
    await client.close()
finally:
    output.cleanup()